# YC Model with ρ (unsquared correlation)\n\nSame mixed-effects model as `2.YC_model.py`, but using ρ_VS (Pearson correlation) instead of ρ²_VS (R²) as the dependent variable.

In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import os
from pathlib import Path
from functools import reduce

from pkg import detrend_group, pearson_corr, flexible_merge, nakagawa_r2

# cd to repo root so ./data/ paths work
REPO = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path(os.getcwd()).resolve().parents[1]
os.chdir(REPO)

### Recompute ρ (unsquared) from yields

In [2]:
yields = pd.read_csv("./data/yields.csv")[["cropname", "year", "yield_og", "csif_og", "whichlag", "country", "iso_a3"]]
yields = detrend_group(yields, 'yield_og', 'yield_log_dt', log_transform=True)
yields = detrend_group(yields, 'csif_og', 'csif_log_dt', log_transform=True)

# ρ instead of ρ²
yield_rho = yields.groupby(['iso_a3', 'cropname', 'whichlag']).apply(
    lambda group: pd.Series({"rho": pearson_corr(group, x="csif_log_dt", y="yield_log_dt")})
).reset_index()
yield_rho = yield_rho[yield_rho['rho'].abs() != 1]
yield_rho.head()

,iso_a3,cropname,whichlag,rho
0,AFG,Barley,yield_log_dt,0.360453
1,AFG,Maize,yield_lag,-0.064995
2,AFG,Millet,yield_lead,0.231947
3,AFG,Potatoes,yield_lag,-0.099211
4,AFG,Pulses nes,yield_lag,0.113851


### Merge covariates

In [3]:
ag = pd.read_csv("./data/ag_vars.csv")[['total_harvarea', 'cropland_fraction', 'cropname', 'iso_a3']]

gdp = pd.read_csv("./data/wb_gdp_per_cap.csv").rename({'value': 'gdp'}, axis=1)
gdp = gdp.groupby(['iso_a3']).agg(avg_gdp=('gdp', 'mean')).reset_index()

flag = pd.read_csv("./data/faostat_all_flags.csv").rename({"Year": 'year'}, axis=1)
flag = flag.groupby(["iso_a3", 'cropname'])[['year']].agg(flag_sum=('year', 'count')).reset_index()

clim = (pd.read_csv("./data/dt_clim_vars.csv").groupby(['iso_a3', 'cropname'])[['sm', 'tmax']]
        .agg(avg_sm=('sm', 'mean'), avg_tmax=('tmax', 'mean')).reset_index())

dfs = [yield_rho, gdp, flag, ag, clim]
merged = reduce(flexible_merge, dfs).dropna(subset=["rho", "avg_sm"]).drop_duplicates().reset_index(drop=True)
merged["flag_sum"] = merged["flag_sum"].fillna(0)

coeffs = ["avg_gdp", "flag_sum", "total_harvarea", "cropland_fraction", "avg_sm", "avg_tmax", "whichlag"]
merged = merged.dropna(subset=coeffs + ["rho", "cropname"]).reset_index(drop=True)
merged.shape

(1744, 10)

### Scale covariates and fit model

In [4]:
df = merged.copy()

coeffs = ["avg_gdp", "flag_sum", "total_harvarea", "cropland_fraction", "avg_sm", "avg_tmax", "whichlag"]
scaled = df[coeffs].drop(columns="whichlag").apply(lambda x: (x - x.mean()) / x.std()).add_suffix("_z")
df["whichlag"] = df["whichlag"].isin(["yield_lag", "yield_lead"]).astype(int)
df = df.drop(columns=df[coeffs].drop(columns="whichlag")).join(scaled)

mod = smf.mixedlm("rho ~ avg_gdp_z + flag_sum_z + whichlag + total_harvarea_z + cropland_fraction_z",
                   df, groups="cropname")
res = mod.fit()
print(res.summary())

             Mixed Linear Model Regression Results
Model:                MixedLM    Dependent Variable:    rho    
No. Observations:     1744       Method:                REML   
No. Groups:           19         Scale:                 0.0552 
Min. group size:      47         Log-Likelihood:        19.2627
Max. group size:      150        Converged:             Yes    
Mean group size:      91.8                                     
---------------------------------------------------------------
                    Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept            0.296    0.012  23.933 0.000  0.272  0.320
avg_gdp_z            0.007    0.006   1.117 0.264 -0.005  0.018
flag_sum_z          -0.028    0.006  -4.363 0.000 -0.040 -0.015
whichlag            -0.339    0.012 -28.516 0.000 -0.362 -0.316
total_harvarea_z     0.022    0.006   3.762 0.000  0.010  0.033
cropland_fraction_z  0.029    0.006   4.849 0.000  0.

/opt/homebrew/Caskroom/miniforge/base/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


### Random effects and fit statistics

In [5]:
sigma2 = float(res.scale)
try:
    tau00 = float(np.asarray(res.cov_re)[0, 0])
except Exception:
    tau00 = np.nan

icc = tau00 / (tau00 + sigma2) if np.isfinite(tau00) else np.nan
r2_marg, r2_cond = nakagawa_r2(res)

print(f"σ²          = {sigma2:.4f}")
print(f"τ₀₀ crop    = {tau00:.4f}")
print(f"ICC         = {icc:.4f}")
print(f"N crops     = {df['cropname'].nunique()}")
print(f"N obs       = {len(res.model.endog)}")
print(f"Marginal R² = {r2_marg:.4f}")
print(f"Conditional R² = {r2_cond:.4f}")

σ²          = 0.0552
τ₀₀ crop    = 0.0015
ICC         = 0.0256
N crops     = 19
N obs       = 1744
Marginal R² = 0.4074
Conditional R² = 0.4226


## Clipped ρ² model\n\nClip negative ρ values to 0, then square to get R². This removes country-crop pairs where the satellite proxy is anti-correlated with survey yields before re-estimating the original R² model.

In [6]:
# Clip negative ρ to 0, then square
merged_clip = merged.copy()
merged_clip['rho_clipped'] = merged_clip['rho'].clip(lower=0)
merged_clip['r2'] = merged_clip['rho_clipped'] ** 2
merged_clip = merged_clip[merged_clip['r2'] != 1].reset_index(drop=True)

print(f"Observations with negative ρ clipped to 0: {(merged['rho'] < 0).sum()} / {len(merged)}")
print(f"Final obs: {len(merged_clip)}")

Observations with negative ρ clipped to 0: 667 / 1744
Final obs: 1744


In [7]:
df2 = merged_clip.copy()

coeffs = ["avg_gdp", "flag_sum", "total_harvarea", "cropland_fraction", "avg_sm", "avg_tmax", "whichlag"]
scaled2 = df2[coeffs].drop(columns="whichlag").apply(lambda x: (x - x.mean()) / x.std()).add_suffix("_z")
df2["whichlag"] = df2["whichlag"].isin(["yield_lag", "yield_lead"]).astype(int)
df2 = df2.drop(columns=df2[coeffs].drop(columns="whichlag")).join(scaled2)

mod2 = smf.mixedlm("r2 ~ avg_gdp_z + flag_sum_z + whichlag + total_harvarea_z + cropland_fraction_z",
                    df2, groups="cropname")
res2 = mod2.fit()
print(res2.summary())

             Mixed Linear Model Regression Results
Model:                MixedLM   Dependent Variable:   r2       
No. Observations:     1744      Method:               REML     
No. Groups:           19        Scale:                0.0160   
Min. group size:      47        Log-Likelihood:       1092.2953
Max. group size:      150       Converged:            Yes      
Mean group size:      91.8                                     
---------------------------------------------------------------
                    Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept            0.153    0.008  19.462 0.000  0.138  0.168
avg_gdp_z            0.008    0.003   2.492 0.013  0.002  0.014
flag_sum_z          -0.009    0.003  -2.751 0.006 -0.016 -0.003
whichlag            -0.128    0.006 -20.014 0.000 -0.141 -0.116
total_harvarea_z     0.014    0.003   4.547 0.000  0.008  0.020
cropland_fraction_z  0.021    0.003   6.612 0.000  0.

/opt/homebrew/Caskroom/miniforge/base/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [8]:
sigma2_2 = float(res2.scale)
try:
    tau00_2 = float(np.asarray(res2.cov_re)[0, 0])
except Exception:
    tau00_2 = np.nan

icc_2 = tau00_2 / (tau00_2 + sigma2_2) if np.isfinite(tau00_2) else np.nan
r2_marg_2, r2_cond_2 = nakagawa_r2(res2)

print(f"σ²             = {sigma2_2:.4f}")
print(f"τ₀₀ crop       = {tau00_2:.4f}")
print(f"ICC            = {icc_2:.4f}")
print(f"N crops        = {df2['cropname'].nunique()}")
print(f"N obs          = {len(res2.model.endog)}")
print(f"Marginal R²    = {r2_marg_2:.4f}")
print(f"Conditional R² = {r2_cond_2:.4f}")

σ²             = 0.0160
τ₀₀ crop       = 0.0008
ICC            = 0.0450
N crops        = 19
N obs          = 1744
Marginal R²    = 0.3041
Conditional R² = 0.3355


# facking fix flags

In [13]:
import pandas as pd

fao_new = pd.read_csv("./data/UNUSED/FAOSTAT_data_en_4-15-2026.csv")

item_to_cropname = {
    'Barley': 'Barley',
    'Maize (corn)': 'Maize',
    'Millet': 'Millet',
    'Potatoes': 'Potatoes',
    'Other pulses n.e.c.': 'Pulses nes',
    'Rice': 'Rice, paddy',
    'Seed cotton, unginned': 'Seed cotton',
    'Sugar beet': 'Sugar beet',
    'Sunflower seed': 'Sunflower seed',
    'Wheat': 'Wheat',
    'Cassava, fresh': 'Cassava',
    'Groundnuts, excluding shelled': 'Groundnuts, with shell',
    'Sorghum': 'Sorghum',
    'Soya beans': 'Soybeans',
    'Sweet potatoes': 'Sweet potatoes',
    'Oats': 'Oats',
    'Rye': 'Rye',
    'Rape or colza seed': 'Rapeseed',
    'Yams': 'Yams',
}

fao_new['cropname'] = fao_new['Item'].map(item_to_cropname)
fao_new = fao_new.rename(columns={'Area Code (ISO3)': 'iso_a3'})

fao_new['is_estimated'] = (fao_new['Flag'] == 'E').astype(int)

flag_pct = (fao_new.groupby(['iso_a3', 'cropname'])
            .agg(n_total=('Year', 'nunique'),
                 n_flagged=('is_estimated', 'sum'))
            .reset_index())
flag_pct['flag_pct'] = flag_pct['n_flagged'] / flag_pct['n_total']

flag_pct.to_csv("./data/fao_flag_pct.csv", index=False)
